# Load the real GTA data

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

STORE_PATH = Path(
    "data/external/gta_store_context_v1.parquet"
)

DEMO_PATH = Path(
    "data/external/statscan/gta_store_demographics_v1.parquet"
)

WEATHER_PATH = Path(
    "data/external/weather/toronto_pearson_weather_daily.parquet"
)

SYNTHETIC_DIR = Path("data/synthetic")
SYNTHETIC_DIR.mkdir(parents=True, exist_ok=True)

stores = pd.read_parquet(STORE_PATH)
demographics = pd.read_parquet(DEMO_PATH)
weather = pd.read_parquet(WEATHER_PATH)

stores["store_id"] = stores["store_id"].astype(str)
demographics["store_id"] = demographics["store_id"].astype(str)

store_master = stores.merge(
    demographics,
    on="store_id",
    how="left",
    suffixes=("", "_demo")
)

print("Stores:", store_master["store_id"].nunique())
display(store_master.head())

Stores: 15


,store_id,store_name,address,city,province,postal_code,full_address,latitude,longitude,universities,...,commercial,retail_landuse,rail_stations,store_name_demo,population_5km,population_density_5km,median_household_income_5km,average_household_size_5km,employment_rate_5km,n_census_tracts
0,3106,Toronto Dufferin Mall,900 Dufferin St,Toronto,ON,M6H 4A9,"900 Dufferin St, Toronto, ON M6H 4A9, Canada",43.655980,-79.435728,19,...,172,519,44,Toronto Dufferin Mall,593613.628819,15197.462826,86252.286112,2.017781,62.389316,147
1,1139,Toronto Downsview,3757 Keele St,Toronto,ON,M3J 1N4,"3757 Keele St, Toronto, ON M3J 1N4, Canada",43.757509,-79.489316,3,...,82,216,18,Toronto Downsview,237963.886322,5979.220710,80848.194959,2.764884,51.253500,63
2,1004,Toronto Stockyards,2525 St Clair Ave W,Toronto,ON,M6N 4Z5,"2525 St Clair Ave W, Toronto, ON M6N 4Z5, Canada",43.667987,-79.484753,0,...,145,498,24,Toronto Stockyards,423374.384031,8590.969933,92072.905741,2.383569,56.936454,123
3,1150,Toronto Downtown East,1000 Gerrard St E,Toronto,ON,M4M 3G6,"1000 Gerrard St E, Toronto, ON M4M 3G6, Canada",43.669300,-79.339861,20,...,108,424,36,Toronto Downtown East,482564.923619,16864.257792,88944.349902,2.079208,59.227363,118
4,3105,Toronto North Park,1305 Lawrence Ave W,Toronto,ON,M6L 1A5,"1305 Lawrence Ave W, Toronto, ON M6L 1A5, Canada",43.708480,-79.473550,1,...,128,427,17,Toronto North Park,373912.266144,6242.331511,85562.143106,2.569160,53.592422,100


# Create 300 synthetic products

In [2]:
CATEGORY_CONFIG = {
    "Hot Beverages": {
        "price": (4, 18),
        "margin": (0.22, 0.38),
        "demand": (5, 18)
    },

    "Cold Beverages": {
        "price": (2, 15),
        "margin": (0.20, 0.35),
        "demand": (8, 24)
    },

    "Frozen Food": {
        "price": (4, 20),
        "margin": (0.18, 0.32),
        "demand": (5, 16)
    },

    "Snacks": {
        "price": (2, 12),
        "margin": (0.25, 0.40),
        "demand": (10, 30)
    },

    "Household Staples": {
        "price": (3, 25),
        "margin": (0.15, 0.28),
        "demand": (8, 22)
    },

    "Personal Care": {
        "price": (4, 25),
        "margin": (0.28, 0.45),
        "demand": (4, 14)
    },

    "Cleaning": {
        "price": (3, 20),
        "margin": (0.20, 0.35),
        "demand": (5, 16)
    },

    "Breakfast": {
        "price": (3, 18),
        "margin": (0.20, 0.35),
        "demand": (6, 20)
    },

    "Ready-to-Eat": {
        "price": (4, 18),
        "margin": (0.22, 0.38),
        "demand": (7, 22)
    },

    "Dairy": {
        "price": (2, 15),
        "margin": (0.15, 0.28),
        "demand": (8, 24)
    }
}

In [3]:
products = []

product_number = 1

for category, config in CATEGORY_CONFIG.items():

    for i in range(30):

        regular_price = rng.uniform(
            *config["price"]
        )

        margin_rate = rng.uniform(
            *config["margin"]
        )

        base_weekly_demand = rng.uniform(
            *config["demand"]
        )

        standard_discount_pct = rng.choice(
        [5, 10, 15, 20, 25]
        )

        vendor_funding_rate = rng.uniform(
        0.45,
        0.85
        )

        products.append({
            "product_id": f"SYN_{product_number:04d}",
            "product_name": f"{category} Product {i+1:02d}",
            "category": category,
            "regular_price": round(regular_price, 2),
            "margin_rate": margin_rate,
            "base_weekly_demand": base_weekly_demand,
            "standard_discount_pct": standard_discount_pct,
            "product_origin": "synthetic",
            "vendor_funding_rate": vendor_funding_rate
        })

        product_number += 1

products = pd.DataFrame(products)

print("Products:", len(products))
print(products["category"].value_counts())

display(products.head())

Products: 300
category
Hot Beverages        30
Cold Beverages       30
Frozen Food          30
Snacks               30
Household Staples    30
Personal Care        30
Cleaning             30
Breakfast            30
Ready-to-Eat         30
Dairy                30
Name: count, dtype: int64


,product_id,product_name,category,regular_price,margin_rate,base_weekly_demand,standard_discount_pct,product_origin,vendor_funding_rate
0,SYN_0001,Hot Beverages Product 01,Hot Beverages,14.84,0.290221,16.161773,5,synthetic,0.487671
1,SYN_0002,Hot Beverages Product 02,Hot Beverages,17.66,0.341782,15.218836,20,synthetic,0.501245
2,SYN_0003,Hot Beverages Product 03,Hot Beverages,10.31,0.279328,17.047945,20,synthetic,0.779105
3,SYN_0004,Hot Beverages Product 04,Hot Beverages,10.21,0.256358,12.209602,20,synthetic,0.475527
4,SYN_0005,Hot Beverages Product 05,Hot Beverages,15.59,0.321066,14.855141,20,synthetic,0.838279


In [4]:
PRODUCT_PATH = (
    SYNTHETIC_DIR /
    "product_catalog_v1.parquet"
)

products.to_parquet(
    PRODUCT_PATH,
    index=False
)

print("Saved:", PRODUCT_PATH)

Saved: data/synthetic/product_catalog_v1.parquet


# Create 104 weeks of real weather

In [5]:
weather["date"] = pd.to_datetime(
    weather["date"]
)

# Data cutoff for this project
DATA_CUTOFF = pd.Timestamp("2026-09-30")

# Only use COMPLETE weeks.
# Sep 30, 2026 is Wednesday, so the last complete week
# ended Sunday Sep 27 and started Monday Sep 21.
last_complete_sunday = (
    DATA_CUTOFF
    - pd.Timedelta(
        days=DATA_CUTOFF.dayofweek + 1
    )
)

weather = weather[
    weather["date"] <= last_complete_sunday
].copy()

# Keep only days containing actual observations
weather = weather[
    weather[
        [
            "mean_temperature",
            "min_temperature",
            "max_temperature",
            "total_precipitation"
        ]
    ]
    .notna()
    .any(axis=1)
].copy()

weather["week_start"] = (
    weather["date"]
    - pd.to_timedelta(
        weather["date"].dt.dayofweek,
        unit="D"
    )
)

weekly_weather = (
    weather
    .groupby("week_start")
    .agg(
        mean_temperature=(
            "mean_temperature",
            "mean"
        ),

        total_rain=(
            "total_rain",
            lambda x: x.sum(
                min_count=1
            )
        ),

        total_snow=(
            "total_snow",
            lambda x: x.sum(
                min_count=1
            )
        )
    )
    .reset_index()
    .sort_values("week_start")
    .tail(104)
)

print(
    "Weather weeks:",
    weekly_weather["week_start"].min(),
    "→",
    weekly_weather["week_start"].max()
)

print(
    "Weeks:",
    len(weekly_weather)
)

Weather weeks: 2024-09-16 00:00:00 → 2026-09-07 00:00:00
Weeks: 104


# Normalize the 5 km area features

In [6]:
def zscore(series):

    x = pd.to_numeric(
        series,
        errors="coerce"
    )

    std = x.std()

    if pd.isna(std) or std == 0:
        return pd.Series(
            0,
            index=series.index
        )

    return (
        (x - x.mean()) / std
    ).fillna(0)

In [7]:
area_features = [
    "population_5km",
    "median_household_income_5km",
    "offices",
    "commercial",
    "universities",
    "colleges"
]

for col in area_features:

    if col not in store_master.columns:
        store_master[col] = 0

    store_master[f"{col}_z"] = zscore(
        store_master[col]
    )

# Create store × product × week data

In [8]:
store_small = store_master[
    [
        "store_id",
        "store_name",
        "city",

        "population_5km",
        "median_household_income_5km",

        "offices",
        "commercial",
        "universities",
        "colleges",

        "population_5km_z",
        "median_household_income_5km_z",
        "offices_z",
        "commercial_z",
        "universities_z",
        "colleges_z"
    ]
].copy()

In [9]:
store_small["_key"] = 1
products["_key"] = 1

panel = store_small.merge(
    products,
    on="_key"
)

panel.drop(
    columns="_key",
    inplace=True
)

weeks = weekly_weather.copy()
weeks["_key"] = 1
panel["_key"] = 1

panel = panel.merge(
    weeks,
    on="_key"
)

panel.drop(
    columns="_key",
    inplace=True
)

print(panel.shape)

(468000, 28)


# Build normal demand

In [10]:
panel["store_demand_factor"] = (
    1
    + 0.10 * panel["population_5km_z"]
    + 0.05 * panel["commercial_z"]
)

panel["store_demand_factor"] = (
    panel["store_demand_factor"]
    .clip(0.65, 1.50)
)

In [11]:
panel["month"] = (
    panel["week_start"].dt.month
)

panel["season_factor"] = 1.0

In [12]:
panel.loc[
    (panel["category"] == "Hot Beverages")
    & panel["month"].isin([11, 12, 1, 2]),
    "season_factor"
] += 0.20

In [13]:
panel.loc[
    (panel["category"] == "Cold Beverages")
    & panel["month"].isin([6, 7, 8]),
    "season_factor"
] += 0.25

In [14]:
panel.loc[
    (panel["category"] == "Frozen Food")
    & panel["month"].isin([6, 7, 8]),
    "season_factor"
] += 0.10

In [15]:
panel["baseline_expected_units"] = (
    panel["base_weekly_demand"]
    * panel["store_demand_factor"]
    * panel["season_factor"]
)

# Create the true promotion effect

In [16]:
panel["true_uplift_rate"] = (
    0.03
    + 0.012 * panel["standard_discount_pct"]
)

In [17]:
panel.loc[
    (panel["category"] == "Hot Beverages")
    & (panel["mean_temperature"] < 5),
    "true_uplift_rate"
] += 0.10

In [18]:
panel.loc[
    (panel["category"] == "Cold Beverages")
    & (panel["mean_temperature"] > 22),
    "true_uplift_rate"
] += 0.12

In [19]:
panel.loc[
    (panel["category"] == "Frozen Food")
    & (panel["mean_temperature"] > 20),
    "true_uplift_rate"
] += 0.05

In [20]:
mask = (
    panel["category"] == "Ready-to-Eat"
)

panel.loc[
    mask,
    "true_uplift_rate"
] += (
    0.04
    * panel.loc[
        mask,
        "offices_z"
    ].clip(-2, 2)
)

In [21]:
student_score = (
    panel["universities_z"]
    + panel["colleges_z"]
) / 2

mask = panel["category"] == "Snacks"

panel.loc[
    mask,
    "true_uplift_rate"
] += (
    0.03
    * student_score[mask]
    .clip(-2, 2)
)

In [22]:
mask = (
    panel["category"]
    == "Household Staples"
)

panel.loc[
    mask,
    "true_uplift_rate"
] += (
    0.03
    * panel.loc[
        mask,
        "population_5km_z"
    ].clip(-2, 2)
)

In [23]:
mask = (
    panel["category"]
    == "Personal Care"
)

panel.loc[
    mask,
    "true_uplift_rate"
] += (
    0.02
    * panel.loc[
        mask,
        "median_household_income_5km_z"
    ].clip(-2, 2)
)

In [24]:
panel["true_uplift_rate"] = (
    panel["true_uplift_rate"]
    .clip(-0.05, 0.65)
)

# Generate promotion assignment

In [25]:
promo_probability = (
    0.15
    + 0.05
    * panel["commercial_z"]
    .clip(-1, 1)
)

promo_probability = (
    promo_probability
    .clip(0.05, 0.35)
)

panel["promotion"] = rng.binomial(
    1,
    promo_probability
)

In [26]:
panel["discount_pct"] = (
    panel["promotion"]
    * panel["standard_discount_pct"]
)

panel["selling_price"] = np.where(
    panel["promotion"] == 1,

    panel["regular_price"]
    * (
        1
        - panel["standard_discount_pct"] / 100
    ),

    panel["regular_price"]
)

# Generate sales

In [27]:
panel["expected_units"] = (
    panel["baseline_expected_units"]
    * (
        1
        + panel["promotion"]
        * panel["true_uplift_rate"]
    )
)

In [28]:
noise = rng.lognormal(
    mean=0,
    sigma=0.15,
    size=len(panel)
)

demand_lambda = (
    panel["expected_units"]
    * noise
)

panel["units_demanded"] = rng.poisson(
    demand_lambda.clip(lower=0.1)
)

In [29]:
panel["inventory"] = np.ceil(
    panel["baseline_expected_units"]
    * rng.uniform(
        1.1,
        1.8,
        size=len(panel)
    )
).astype(int)

In [30]:
panel["units_sold"] = np.minimum(
    panel["units_demanded"],
    panel["inventory"]
)

# Revenue and profit

In [31]:
panel["revenue"] = (
    panel["units_sold"]
    * panel["selling_price"]
)

In [32]:
panel["unit_cost"] = (
    panel["regular_price"]
    * (
        1 - panel["margin_rate"]
    )
)

In [33]:
panel["promo_funding_per_unit"] = (
    panel["promotion"]
    * panel["regular_price"]
    * (panel["discount_pct"] / 100)
    * panel["vendor_funding_rate"]
)

panel["effective_unit_cost"] = (
    panel["unit_cost"]
    - panel["promo_funding_per_unit"]
)

panel["unit_margin"] = (
    panel["selling_price"]
    - panel["effective_unit_cost"]
)

panel["profit"] = (
    panel["units_sold"]
    * panel["unit_margin"]
)

# Store the true causal effect

In [34]:
panel["expected_profit_control"] = (
    panel["baseline_expected_units"]
    * (
        panel["regular_price"]
        - panel["unit_cost"]
    )
)

In [35]:
promo_price = (
    panel["regular_price"]
    * (
        1
        - panel["standard_discount_pct"] / 100
    )
)

In [36]:
promo_price = (
    panel["regular_price"]
    * (
        1
        - panel["standard_discount_pct"] / 100
    )
)

treated_unit_cost = (
    panel["unit_cost"]
    - (
        panel["regular_price"]
        * (panel["standard_discount_pct"] / 100)
        * panel["vendor_funding_rate"]
    )
)

panel["expected_profit_control"] = (
    panel["baseline_expected_units"]
    * (
        panel["regular_price"]
        - panel["unit_cost"]
    )
)

panel["expected_profit_treated"] = (
    panel["baseline_expected_units"]
    * (1 + panel["true_uplift_rate"])
    * (
        promo_price
        - treated_unit_cost
    )
)

panel["true_incremental_profit"] = (
    panel["expected_profit_treated"]
    - panel["expected_profit_control"]
)

panel["true_incremental_units"] = (
    panel["baseline_expected_units"]
    * panel["true_uplift_rate"]
)

In [37]:
panel["true_incremental_profit"] = (
    panel["expected_profit_treated"]
    - panel["expected_profit_control"]
)

In [38]:
panel["true_incremental_units"] = (
    panel["baseline_expected_units"]
    * panel["true_uplift_rate"]
)

# Mark what is real vs synthetic

In [39]:
panel["store_context_origin"] = "real"
panel["weather_origin"] = "real"
panel["product_origin"] = "synthetic"
panel["sales_outcome_origin"] = "synthetic"
panel["simulation_version"] = "v1"

In [40]:
print("Rows:", len(panel))
print("Stores:", panel["store_id"].nunique())
print("Products:", panel["product_id"].nunique())
print("Weeks:", panel["week_start"].nunique())

print(
    "Promotion rate:",
    round(
        panel["promotion"].mean() * 100,
        2
    ),
    "%"
)

print(
    "Average units:",
    round(
        panel["units_sold"].mean(),
        2
    )
)

print(
    "True uplift range:",
    round(
        panel["true_uplift_rate"].min(),
        3
    ),
    "→",
    round(
        panel["true_uplift_rate"].max(),
        3
    )
)

print(
    "Positive incremental-profit rows:",
    round(
        (
            panel["true_incremental_profit"] > 0
        ).mean() * 100,
        2
    ),
    "%"
)

Rows: 468000
Stores: 15
Products: 300
Weeks: 104
Promotion rate: 14.78 %
Average units: 13.82
True uplift range: 0.046 → 0.45
Positive incremental-profit rows: 49.86 %


In [41]:
HISTORY_PATH = (
    SYNTHETIC_DIR /
    "gta_store_product_week_history_v1.parquet"
)

panel.to_parquet(
    HISTORY_PATH,
    index=False
)

print("Saved:", HISTORY_PATH)

Saved: data/synthetic/gta_store_product_week_history_v1.parquet
